# SCADA Flow — Pipeline Orchestration Guide

**mcom technology** | © 2026 All rights reserved.

## Architecture Overview

```
┌──────────────────────────────────────────────────────────────────┐
│  LOCAL MACHINE (Windows Task Scheduler / cron)             │
│                                                            │
│  scada_flow_ingestion_local.py                             │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │ 1. Connect to SQL Server (pymssql)                      │  │
│  │ 2. Read tables in 500K-row chunks -> Parquet            │  │
│  │ 3. Upload Parquet to UC Volume (4 parallel threads)     │  │
│  │ 4. CREATE TABLE via read_files() -> Delta                │  │
│  │ 5. Verify row counts                                    │  │
│  │ 6. Trigger Databricks Job via REST API  ──────────┐     │  │
│  └──────────────────────────────────────────┴───────────────┘  │
└────────────────────────────────────────────────────────┴─────────┘
                                                             │
                                                    REST API trigger
                                                             │
┌──────────────────────────────────────────────────────────────────┐
│  DATABRICKS (Lakeflow Job)                                  │
│                                                            │
│  Job: SCADA Flow - Data Pipeline (24h refresh)             │
│  Job ID: 127178333405535                                   │
│  Schedule: Daily at 6:00 AM UTC (11:30 AM IST)             │
│                                                            │
│  03_Data_Pipeline notebook:                                │
│  ┌──────────────────────────────────────────────────────────┐  │
│  │ 1. Refresh device_list                                │  │
│  │ 2. Refresh device_parameters                          │  │
│  │ 3. Refresh sensor_log (dedup + clean, ~27M rows)       │  │
│  │ 4. Refresh alerts (HTML strip + regex extract)         │  │
│  │ 5. Refresh totalizer (PNAME normalize)                 │  │
│  │ 6. Data quality validation                             │  │
│  │ 7. Log results to workspace.scada_pipeline.run_log     │  │
│  └──────────────────────────────────────────────────────────┘  │
│                                                            │
│  Dashboard auto-refreshes from workspace.scada_cleaned     │
└──────────────────────────────────────────────────────────────────┘
```

## Two Trigger Modes

### Mode 1: Event-Driven (after local ingestion)
The local `scada_flow_ingestion_local.py` script triggers the Databricks Job via REST API after it finishes uploading data. This is the **primary mode**.

### Mode 2: Scheduled (backup/standalone)
The Databricks Job also runs on a daily schedule (6 AM UTC / 11:30 AM IST). This acts as a **safety net** in case the local script fails to trigger, or if you need to re-run cleaning independently.

---

## Setup Instructions

### 1. Windows Task Scheduler (for daily local ingestion)

Create a scheduled task on your local machine:

```powershell
# Open Task Scheduler or use PowerShell:
schtasks /create /tn "SCADA_Flow_Ingestion" /tr "python C:\path\to\scada_flow_ingestion_local.py" /sc daily /st 05:00 /rl HIGHEST
```

Or use the Task Scheduler GUI:
- **Name:** SCADA_Flow_Ingestion
- **Trigger:** Daily at 5:00 AM IST (before the Databricks backup schedule at 11:30 AM IST)
- **Action:** `python scada_flow_ingestion_local.py`
- **Settings:** Run whether user is logged on or not

### 2. Environment Variables (recommended for security)

```bash
set DATABRICKS_HOST=https://dbc-526464cb-e1ae.cloud.databricks.com
set DATABRICKS_TOKEN=<your-personal-access-token>
set SQL_USER=tushar89
set SQL_PASS=<your-sql-password>
```

### 3. Unpause the Databricks Job

The Job is created in **PAUSED** state. When ready to go live:
- Go to the Jobs page
- Find "SCADA Flow - Data Pipeline (24h refresh)"
- Click the schedule toggle to unpause

---

## Job Details

| Property | Value |
|---|---|
| Job Name | SCADA Flow - Data Pipeline (24h refresh) |
| Job ID | 127178333405535 |
| Notebook | `/Users/.../SCADA_Flow_Monitoring/03_Data_Pipeline` |
| Schedule | Daily at 6:00 AM UTC (11:30 AM IST) |
| Compute | Serverless |
| Max Concurrent Runs | 1 |
| Tags | project=scada_flow, owner=mcom_technology |

---

## Monitoring

### Pipeline Run Log
Every run logs steps to `workspace.scada_pipeline.run_log`:

```sql
SELECT * FROM workspace.scada_pipeline.run_log
ORDER BY completed_at DESC
LIMIT 20
```

### Data Quality Checks
The pipeline validates:
- **Row count thresholds** — sensor_log > 25M, device_list > 500, etc.
- **Data freshness** — latest sensor reading should be within 48 hours

### Failure Handling
- Each cleaning step is wrapped in try/except
- Failures are logged to the audit table with error messages
- The Job fails fast on any step failure (raises exception)
- Databricks sends email notification on failure (configurable)

---

## Data Flow Timeline (Daily)

| Time (IST) | Action |
|---|---|
| 5:00 AM | Windows Task Scheduler triggers local ingestion script |
| 5:00-5:30 AM | Script reads SQL Server, uploads Parquet to UC Volume, creates raw Delta tables |
| 5:30 AM | Script triggers Databricks Job via REST API |
| 5:30-5:45 AM | Databricks Job runs: clean 5 tables, validate, log results |
| 5:45 AM+ | Dashboard queries run against refreshed `workspace.scada_cleaned` tables |
| 11:30 AM | Backup schedule runs (if ingestion trigger was missed) |

---

*© 2026 mcom technology. All rights reserved.*

In [0]:
%sql
-- Recent pipeline run history
SELECT
  run_id,
  step_name,
  status,
  rows_affected,
  duration_seconds,
  completed_at,
  error_message
FROM workspace.scada_pipeline.run_log
ORDER BY completed_at DESC
LIMIT 30

In [0]:
%sql
-- How fresh is the data in each cleaned table?
SELECT 'sensor_log' as table_name,
  COUNT(*) as total_rows,
  MAX(data_timestamp) as latest_record,
  DATEDIFF(HOUR, MAX(data_timestamp), CURRENT_TIMESTAMP()) as hours_stale
FROM workspace.scada_cleaned.sensor_log
UNION ALL
SELECT 'alerts', COUNT(*), MAX(report_date), DATEDIFF(HOUR, MAX(report_date), CURRENT_TIMESTAMP())
FROM workspace.scada_cleaned.alerts
UNION ALL
SELECT 'totalizer', COUNT(*), MAX(report_date), DATEDIFF(HOUR, MAX(report_date), CURRENT_TIMESTAMP())
FROM workspace.scada_cleaned.totalizer
UNION ALL
SELECT 'device_list', COUNT(*), NULL, NULL
FROM workspace.scada_cleaned.device_list
UNION ALL
SELECT 'device_parameters', COUNT(*), NULL, NULL
FROM workspace.scada_cleaned.device_parameters

In [0]:
# Uncomment and run to manually trigger the pipeline Job
# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
# run = w.jobs.run_now(127178333405535, notebook_params={"run_id": f"manual_{int(__import__('time').time())}"})
# print(f"Triggered! Run ID: {run.run_id}")
# print(f"Monitor at: /jobs/127178333405535/runs/{run.run_id}")